# Notebook 16 – Feature Engineering Mini Challenge

## Objective

In this challenge, we independently analyze the Titanic dataset and perform feature engineering.

The main tasks are:

1. Understand the dataset.
2. Identify the target.
3. Identify existing features.
4. Identify useful new features.
5. Create at least 10 meaningful features.
6. Explain every feature.
7. Check for leakage.
8. Remove irrelevant features.
9. Perform feature selection.
10. Compare the dataset before and after feature engineering.

The objective is to demonstrate independent problem-solving and domain-based feature engineering.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Understand the Dataset

The Titanic dataset contains passenger information such as age, passenger class, gender, family information, fare, ticket, and port of embarkation.

The target variable is `Survived`.

- 0 → Did not survive
- 1 → Survived

In [2]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())

Rows: 891
Columns: 12

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


## 2. Identify the Target

The target variable is `Survived`.

It represents whether the passenger survived the Titanic disaster.

The remaining useful columns are considered input features for prediction.

In [3]:
target = "Survived"

print("Target variable:", target)
print("\nTarget distribution:")
print(df[target].value_counts())

Target variable: Survived

Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64


## 3. Identify Existing Features

Important existing features include:

- Pclass – Passenger class
- Sex – Passenger gender
- Age – Passenger age
- SibSp – Siblings or spouses aboard
- Parch – Parents or children aboard
- Fare – Ticket fare
- Embarked – Port of embarkation
- Ticket – Ticket identifier
- Name – Passenger name

These features can be transformed into more meaningful information.

In [4]:
existing_features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch",
    "Fare", "Embarked", "Ticket", "Name"
]

print("Existing features:")
print(existing_features)

Existing features:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Ticket', 'Name']


## 4. Compare Dataset Before Feature Engineering

Before creating new features, we record the original dataset shape.

This gives us a simple baseline for comparing the dataset after feature engineering.

In [5]:
before_shape = df.shape

print("Before Feature Engineering")
print("Rows:", before_shape[0])
print("Columns:", before_shape[1])

Before Feature Engineering
Rows: 891
Columns: 12


## 5. Create Numerical Features

Numerical features can be created by combining existing numerical columns.

### Feature 1 – FamilySize
Formula: `SibSp + Parch + 1`

It represents the total family size of the passenger.

### Feature 2 – IsAlone
Value is 1 when FamilySize is 1, otherwise 0.

### Feature 3 – FarePerPerson
Formula: `Fare / FamilySize`

It represents the approximate fare per family member.

In [6]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

print(df[[
    "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "FarePerPerson"
]].head())

   SibSp  Parch     Fare  FamilySize  IsAlone  FarePerPerson
0      1      0   7.2500           2        0        3.62500
1      1      0  71.2833           2        0       35.64165
2      0      0   7.9250           1        1        7.92500
3      1      0  53.1000           2        0       26.55000
4      0      0   8.0500           1        1        8.05000


## 6. Create Additional Numerical Features

### Feature 4 – FamilyDifference
Formula: `SibSp - Parch`

It represents the difference between siblings/spouses and parents/children.

### Feature 5 – AbsoluteFamilyDifference
Formula: `abs(SibSp - Parch)`

It represents the absolute difference between the two family-related counts.

### Feature 6 – AgeSquared
Formula: `Age²`

It provides a non-linear representation of age.

In [7]:
df["FamilyDifference"] = df["SibSp"] - df["Parch"]
df["AbsoluteFamilyDifference"] = abs(df["FamilyDifference"])
df["AgeSquared"] = df["Age"] ** 2

print(df[[
    "SibSp", "Parch", "FamilyDifference",
    "AbsoluteFamilyDifference", "Age", "AgeSquared"
]].head())

   SibSp  Parch  FamilyDifference  AbsoluteFamilyDifference   Age  AgeSquared
0      1      0                 1                         1  22.0       484.0
1      1      0                 1                         1  38.0      1444.0
2      0      0                 0                         0  26.0       676.0
3      1      0                 1                         1  35.0      1225.0
4      0      0                 0                         0  35.0      1225.0


## 7. Create Categorical Features

### Feature 7 – Title
Extract the passenger's title from the Name column.

### Feature 8 – AgeGroup
Convert age into meaningful groups:
- Child
- Teenager
- Young Adult
- Adult
- Senior

These features provide more understandable demographic information.

In [8]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False)

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 17, 30, 50, np.inf],
    labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"]
)

print(df[["Name", "Title", "Age", "AgeGroup"]].head())

                                                Name Title   Age     AgeGroup
0                            Braund, Mr. Owen Harris    Mr  22.0  Young Adult
1  Cumings, Mrs. John Bradley (Florence Briggs Th...   Mrs  38.0        Adult
2                             Heikkinen, Miss. Laina  Miss  26.0  Young Adult
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)   Mrs  35.0        Adult
4                           Allen, Mr. William Henry    Mr  35.0        Adult


## 8. Create Aggregation Features

### Feature 9 – TicketPassengerCount
Counts how many passengers shared the same ticket.

### Feature 10 – TicketAverageFare
Calculates the average fare for passengers sharing the same ticket.

These features provide group-level information that may be useful for prediction.

In [9]:
df["TicketPassengerCount"] = df.groupby("Ticket")["PassengerId"].transform("count")

df["TicketAverageFare"] = df.groupby("Ticket")["Fare"].transform("mean")

print(df[[
    "Ticket", "TicketPassengerCount", "TicketAverageFare"
]].head())

             Ticket  TicketPassengerCount  TicketAverageFare
0         A/5 21171                     1             7.2500
1          PC 17599                     1            71.2833
2  STON/O2. 3101282                     1             7.9250
3            113803                     2            53.1000
4            373450                     1             8.0500


## 9. Create Interaction Features

### Feature 11 – AgePclassInteraction
Formula: `Age × Pclass`

It combines age and passenger class to represent their interaction.

### Feature 12 – FamilyFareInteraction
Formula: `FamilySize × Fare`

It combines family size and fare information.

Interaction features can help a model capture relationships between multiple variables.

In [10]:
df["AgePclassInteraction"] = df["Age"] * df["Pclass"]
df["FamilyFareInteraction"] = df["FamilySize"] * df["Fare"]

print(df[[
    "Age", "Pclass", "AgePclassInteraction",
    "FamilySize", "Fare", "FamilyFareInteraction"
]].head())

    Age  Pclass  AgePclassInteraction  FamilySize     Fare  \
0  22.0       3                  66.0           2   7.2500   
1  38.0       1                  38.0           2  71.2833   
2  26.0       3                  78.0           1   7.9250   
3  35.0       1                  35.0           2  53.1000   
4  35.0       3                 105.0           1   8.0500   

   FamilyFareInteraction  
0                14.5000  
1               142.5666  
2                 7.9250  
3               106.2000  
4                 8.0500  


## 10. Summary of Created Features

We created 12 meaningful features:

1. FamilySize
2. IsAlone
3. FarePerPerson
4. FamilyDifference
5. AbsoluteFamilyDifference
6. AgeSquared
7. Title
8. AgeGroup
9. TicketPassengerCount
10. TicketAverageFare
11. AgePclassInteraction
12. FamilyFareInteraction

In [11]:
new_features = [
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "FamilyDifference",
    "AbsoluteFamilyDifference",
    "AgeSquared",
    "Title",
    "AgeGroup",
    "TicketPassengerCount",
    "TicketAverageFare",
    "AgePclassInteraction",
    "FamilyFareInteraction"
]

print("Number of engineered features:", len(new_features))
print(new_features)

Number of engineered features: 12
['FamilySize', 'IsAlone', 'FarePerPerson', 'FamilyDifference', 'AbsoluteFamilyDifference', 'AgeSquared', 'Title', 'AgeGroup', 'TicketPassengerCount', 'TicketAverageFare', 'AgePclassInteraction', 'FamilyFareInteraction']


## 11. Check for Feature Leakage

Feature leakage happens when a feature contains information that would not be available at prediction time.

In this challenge:

- None of the engineered features directly use `Survived`.
- FamilySize uses SibSp and Parch.
- Title uses Name.
- TicketPassengerCount uses Ticket information.
- Interaction features use existing input features.

Therefore, these features do not directly contain the target.

However, aggregation features should always be calculated using information available at the prediction time in a real ML workflow.

In [12]:
leakage_features = [
    feature for feature in new_features
    if "Survived" in feature
]

print("Direct target-based features:", leakage_features)

if not leakage_features:
    print("No direct target leakage detected.")

Direct target-based features: []
No direct target leakage detected.


## 12. Remove Irrelevant Features

Some columns are identifiers or contain information that is not directly useful in their raw form.

`PassengerId` is an identifier, so it is removed.

`Name` is also removed after extracting the useful `Title` feature.

The target `Survived` is kept separately and is not used as an input feature.

In [13]:
remove_columns = ["PassengerId", "Name"]

df_model = df.drop(columns=remove_columns)

print("Removed columns:", remove_columns)
print("New shape:", df_model.shape)

Removed columns: ['PassengerId', 'Name']
New shape: (891, 22)


## 13. Prepare Features for Machine Learning

Machine Learning models require numerical input.

Therefore:
- Categorical features are converted using One-Hot Encoding.
- Missing numerical values are filled using the median.
- The target variable is kept separately.

In [14]:
model_features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "FamilySize", "IsAlone", "FarePerPerson",
    "FamilyDifference", "AbsoluteFamilyDifference", "AgeSquared",
    "Title", "AgeGroup", "TicketPassengerCount",
    "TicketAverageFare", "AgePclassInteraction",
    "FamilyFareInteraction"
]

X = df_model[model_features].copy()
y = df_model["Survived"]

X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

print("ML-ready feature shape:", X.shape)
X.head()

ML-ready feature shape: (891, 38)


,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,FarePerPerson,FamilyDifference,AbsoluteFamilyDifference,...,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess,AgeGroup_Teenager,AgeGroup_Young Adult,AgeGroup_Adult,AgeGroup_Senior
0,3,22.0,1,0,7.2500,2,0,3.62500,1,1,...,True,False,False,False,False,False,False,True,False,False
1,1,38.0,1,0,71.2833,2,0,35.64165,1,1,...,False,True,False,False,False,False,False,False,True,False
2,3,26.0,0,0,7.9250,1,1,7.92500,0,0,...,False,False,False,False,False,False,False,True,False,False
3,1,35.0,1,0,53.1000,2,0,26.55000,1,1,...,False,True,False,False,False,False,False,False,True,False
4,3,35.0,0,0,8.0500,1,1,8.05000,0,0,...,True,False,False,False,False,False,False,False,True,False


## 14. Perform Feature Selection

Feature selection identifies useful features and removes unnecessary ones.

A simple Random Forest model can be used to identify features with higher predictive importance.

Feature importance is used as a guide; it does not automatically prove that a feature should be removed.

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(10)

,Feature,Importance
15,Sex_male,0.126842
28,Title_Mr,0.099517
13,AgePclassInteraction,0.089187
7,FarePerPerson,0.086839
14,FamilyFareInteraction,0.073458
4,Fare,0.072463
12,TicketAverageFare,0.070186
10,AgeSquared,0.065346
1,Age,0.060749
0,Pclass,0.030875


## 15. Select Important Features

For this demonstration, we select the top 10 features based on Random Forest importance.

In a real ML project, feature selection should also consider model performance, domain knowledge, correlation, and leakage.

In [16]:
selected_features = importance.head(10)["Feature"].tolist()

X_selected = X[selected_features]

print("Selected features:")
print(selected_features)
print("\nSelected feature shape:", X_selected.shape)

Selected features:
['Sex_male', 'Title_Mr', 'AgePclassInteraction', 'FarePerPerson', 'FamilyFareInteraction', 'Fare', 'TicketAverageFare', 'AgeSquared', 'Age', 'Pclass']

Selected feature shape: (891, 10)


## 16. Compare Dataset Before and After Feature Engineering

Before feature engineering, the dataset contained only the original columns.

After feature engineering, additional numerical, categorical, aggregation, and interaction features were created.

This comparison helps us understand the impact of feature engineering on the dataset structure.

In [17]:
after_shape = df_model.shape

comparison = pd.DataFrame({
    "Stage": ["Before Feature Engineering", "After Feature Engineering"],
    "Rows": [before_shape[0], after_shape[0]],
    "Columns": [before_shape[1], after_shape[1]]
})

comparison

,Stage,Rows,Columns
0,Before Feature Engineering,891,12
1,After Feature Engineering,891,22


## 17. Required Documentation for Important Engineered Features

For each important engineered feature, document:

### Feature Name
Name of the newly created feature.

### Source Columns
Columns used to create the feature.

### Logic
Formula or business rule used.

### Reason
Why the feature was created.

### Business Meaning
What the feature represents.

### ML Relevance
How the feature could help the Machine Learning model.

### Leakage Check
Whether the feature could introduce data leakage.

### Final Decision
- Retain
- Remove
- Needs Further Analysis

In [18]:
documentation = pd.DataFrame([
    ["FamilySize", "SibSp, Parch", "SibSp + Parch + 1",
     "Represent family size", "Total family members",
     "May capture family survival patterns",
     "No target used", "Retain"],

    ["IsAlone", "FamilySize", "FamilySize == 1",
     "Identify solo passengers", "Travelling alone",
     "May capture different survival patterns",
     "No target used", "Retain"],

    ["FarePerPerson", "Fare, FamilySize", "Fare / FamilySize",
     "Represent fare per person", "Approximate individual fare",
     "May provide useful fare information",
     "No target used", "Needs Further Analysis"],

    ["Title", "Name", "Extract title from Name",
     "Capture passenger title", "Social/title information",
     "May provide demographic information",
     "No target used", "Retain"],

    ["AgeGroup", "Age", "Age-based groups",
     "Create meaningful age categories", "Passenger age group",
     "May capture non-linear age patterns",
     "No target used", "Retain"],

    ["TicketPassengerCount", "Ticket, PassengerId",
     "Count passengers per ticket",
     "Capture ticket group size", "Number sharing a ticket",
     "May capture group-related patterns",
     "No target used", "Retain"],

    ["TicketAverageFare", "Ticket, Fare",
     "Mean fare per ticket",
     "Capture ticket-level fare information", "Average fare for ticket group",
     "May represent ticket group characteristics",
     "No target used", "Needs Further Analysis"],

    ["AgePclassInteraction", "Age, Pclass", "Age * Pclass",
     "Capture age-class relationship", "Combined age and class",
     "May capture feature interactions",
     "No target used", "Retain"],

    ["FamilyFareInteraction", "FamilySize, Fare",
     "FamilySize * Fare",
     "Capture family-fare relationship", "Combined family and fare information",
     "May capture non-linear patterns",
     "No target used", "Needs Further Analysis"],

    ["AgeSquared", "Age", "Age ** 2",
     "Represent non-linear age effect", "Squared age value",
     "May help models capture non-linear relationships",
     "No target used", "Needs Further Analysis"]
], columns=[
    "Feature Name",
    "Source Columns",
    "Logic",
    "Reason",
    "Business Meaning",
    "ML Relevance",
    "Leakage Check",
    "Final Decision"
])

documentation

,Feature Name,Source Columns,Logic,Reason,Business Meaning,ML Relevance,Leakage Check,Final Decision
0,FamilySize,"SibSp, Parch",SibSp + Parch + 1,Represent family size,Total family members,May capture family survival patterns,No target used,Retain
1,IsAlone,FamilySize,FamilySize == 1,Identify solo passengers,Travelling alone,May capture different survival patterns,No target used,Retain
2,FarePerPerson,"Fare, FamilySize",Fare / FamilySize,Represent fare per person,Approximate individual fare,May provide useful fare information,No target used,Needs Further Analysis
3,Title,Name,Extract title from Name,Capture passenger title,Social/title information,May provide demographic information,No target used,Retain
4,AgeGroup,Age,Age-based groups,Create meaningful age categories,Passenger age group,May capture non-linear age patterns,No target used,Retain
5,TicketPassengerCount,"Ticket, PassengerId",Count passengers per ticket,Capture ticket group size,Number sharing a ticket,May capture group-related patterns,No target used,Retain
6,TicketAverageFare,"Ticket, Fare",Mean fare per ticket,Capture ticket-level fare information,Average fare for ticket group,May represent ticket group characteristics,No target used,Needs Further Analysis
7,AgePclassInteraction,"Age, Pclass",Age * Pclass,Capture age-class relationship,Combined age and class,May capture feature interactions,No target used,Retain
8,FamilyFareInteraction,"FamilySize, Fare",FamilySize * Fare,Capture family-fare relationship,Combined family and fare information,May capture non-linear patterns,No target used,Needs Further Analysis
9,AgeSquared,Age,Age ** 2,Represent non-linear age effect,Squared age value,May help models capture non-linear relationships,No target used,Needs Further Analysis


## 18. Final Feature Engineering Result

The final workflow is:

**Original Dataset**
→ Understand existing features

**Engineered Dataset**
→ Create numerical, categorical, aggregation, and interaction features

**Selected Features**
→ Select useful features using feature importance

**Final ML-Ready Feature Set**
→ Encode categorical variables and handle missing values

The final dataset contains features that are more informative for Machine Learning while unnecessary columns are removed and leakage is checked.

In [19]:
print("Original dataset shape:", df.shape)
print("Engineered dataset shape:", df_model.shape)
print("ML-ready feature shape:", X.shape)
print("Selected feature shape:", X_selected.shape)

print("\nFeature Engineering Challenge completed successfully.")

Original dataset shape: (891, 24)
Engineered dataset shape: (891, 22)
ML-ready feature shape: (891, 38)
Selected feature shape: (891, 10)

Feature Engineering Challenge completed successfully.


## 19. Conclusion

This challenge demonstrated an independent Feature Engineering workflow using the Titanic dataset.

We identified the target and existing features, created more than 10 meaningful features, checked for leakage, removed irrelevant columns, performed feature selection, and compared the dataset before and after feature engineering.

The main takeaway is:

**Good feature engineering converts raw information into meaningful features that can help a Machine Learning model learn useful patterns.**

Every engineered feature should be evaluated based on its business meaning, ML relevance, leakage risk, and final usefulness.

In [20]:
print("Mini Challenge completed successfully.")
print("Engineered features:", len(new_features))
print("Selected features:", len(selected_features))

Mini Challenge completed successfully.
Engineered features: 12
Selected features: 10
